# Lick figures per animal — corrected reviewer-analysis figures

Rebuilds the `3c_*` reviewer-analysis figures (spatial-vs-temporal strategy,
dark-epoch specificity, screen-transition classification, post-VR-onset
persistence, black-screen speed, trial bootstrap) with the 7 method fixes
(M1-M7) from the second-round critique. All logic lives in
`lick_figures_per_animal.py` (Phase 3 consolidation); this notebook is the
driver that loads data, runs the M1-M7 pipeline, and produces the specified
figures.

**Method fixes** (see the module docstring for full detail):
- **M1** baseline z-score reference = B5-B8 (not B7+B8, not B1-B8)
- **M2** hierarchical (day→trial) bootstrap over B5-B8, trial-clustered,
  10000 iterations, gated by a leave-one-day-out calibration check —
  p-values are only shown where the calibration passes (FPR = 0 across the
  4 held-out days; strict, since any non-zero FPR here is already ≥0.25,
  5× nominal alpha)
- **M3** "spatial" strategy computed both as originally defined (any
  approach-category lick) and restricted (must fall within the final 30 cm
  before the corridor end)
- **M4** drinking bouts split into pre-/post-valve parts; persistence
  measured against the actual next-trial VR-on frame, not a nominal 1.5 s
- **M5** ONE VR running-speed definition (freeze-excluded instantaneous
  mean, per trial) used everywhere; speed-matched lick spread restricts to
  the overlapping per-trial speed range between two sessions
- **M6** deceleration onset reported as distance BEFORE the corridor end
  (higher = earlier deceleration = more anticipatory), removing the sign
  ambiguity
- **M7** approach licks within 10 cm of the corridor start excluded as
  likely previous-trial carryover, on top of the existing time-based fix

**Figure style**: fixed condition colors/order (baseline days grey
gradient, Saline/DCZ100/DCZ200 distinct), every drug-session panel labelled
"n = 1 session", PNG+SVG @ 300 dpi to `<output>/<animal>/`. No p-values
plotted until the M2 calibration check passes for that (animal, metric).

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MODULE_DIR = r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation\09.LickingBehavior"
sys.path.insert(0, MODULE_DIR)
import lick_figures_per_animal as lfa

lfa.set_figure_style()
OUTPUT_ROOT = os.path.join(lfa.DATASET_ROOT, "lick_figures_per_animal")
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print("output ->", OUTPUT_ROOT)

## 1. Load data

In [ ]:
M = pd.read_csv(os.path.join(lfa.DATASET_ROOT, "all_sessions_lickmetrics.csv"))

t0 = time.time()
sessions = {a: {} for a in lfa.ANIMALS}
for _, row in M.iterrows():
    sessions[row.animal][row.session_label] = lfa.load_session_data(row.recording, row.animal)
print(f"loaded {sum(len(v) for v in sessions.values())} sessions in {time.time()-t0:.0f}s")

E = lfa.build_enriched_table(sessions, M)
print("enriched table:", E.shape)
E.head()

## 2. M2 — hierarchical bootstrap + calibration check

Runs BEFORE any figure that would show a p-value. Per (animal, metric):
leave-one-day-out calibration across B5-B8 first; the condition-vs-baseline
bootstrap's p-value is only kept if that calibration's false-positive rate
is exactly 0 (any of the 4 held-out days coming back significant already
means FPR >= 0.25, five times the nominal 0.05 — a real failure, not noise,
even accounting for how coarse a 4-point FPR estimate is).

In [ ]:
BOOT_RESULTS, CALIBRATION = lfa.run_bootstrap_summary(sessions, n_boot=10000, calib_n_boot=2000)

print("=== calibration (false-positive rate per animal x metric; must be 0 to trust p-values) ===")
fpr_table = CALIBRATION.groupby(["animal", "metric"]).significant.mean().rename("fpr").reset_index()
print(fpr_table.to_string(index=False))
print()
print("=== bootstrap results (p_value_plottable is NaN wherever calibration failed) ===")
print(BOOT_RESULTS[["animal", "session_label", "metric", "observed", "p_value", "calibration_fpr", "p_value_plottable"]]
     .round(4).to_string(index=False))

BOOT_RESULTS.to_csv(os.path.join(OUTPUT_ROOT, "bootstrap_results_hierarchical.csv"), index=False)
CALIBRATION.to_csv(os.path.join(OUTPUT_ROOT, "bootstrap_calibration.csv"), index=False)
print()
print("wrote bootstrap_results_hierarchical.csv, bootstrap_calibration.csv ->", OUTPUT_ROOT)

## 3. JSY083 figures (A1-A5)

In [ ]:
fig = lfa.fig_A1_B1_spread_inzone(E, "JSY083")
fig.suptitle("Approach licking narrows toward the reward zone over days 1 to 4", y=1.04)
lfa.save_fig(fig, "JSY083", "A1_spread_inzone_by_day", OUTPUT_ROOT)
plt.show()

In [ ]:
fig = lfa.fig_A2_strategy_by_day(E, "JSY083")
fig.suptitle("After day 4, anticipation shifts to the screen transition", y=1.04)
lfa.save_fig(fig, "JSY083", "A2_strategy_by_day", OUTPUT_ROOT)
plt.show()

In [ ]:
fig = lfa.fig_A3_B3_dcz_spread_inzone(E, sessions, "JSY083")
fig.suptitle("Under DCZ, approach licking broadens and moves away from the reward zone", y=1.04)
lfa.save_fig(fig, "JSY083", "A3_dcz_spread_inzone", OUTPUT_ROOT)
plt.show()

In [ ]:
fig = lfa.fig_persistence(E, "JSY083")
fig.suptitle("Drinking continues into the next trial under DCZ", y=1.04)
lfa.save_fig(fig, "JSY083", "A4_post_vr_persistence", OUTPUT_ROOT)
plt.show()

In [ ]:
fig = lfa.fig_dark_anticipation(E, "JSY083")
fig.suptitle("Reward expectation in the dark is preserved", y=1.04)
lfa.save_fig(fig, "JSY083", "A5_dark_anticipation", OUTPUT_ROOT)
plt.show()

## 4. JSY084 figures (B1-B4)

In [ ]:
fig = lfa.fig_A1_B1_spread_inzone(E, "JSY084")
fig.suptitle("Approach licking improves early and the strategy mix stays stable (1/2)", y=1.04)
lfa.save_fig(fig, "JSY084", "B1a_spread_inzone_by_day", OUTPUT_ROOT)
plt.show()

fig2 = lfa.fig_A2_strategy_by_day(E, "JSY084", sessions=tuple(lfa.BASELINE_DAYS))
fig2.suptitle("Approach licking improves early and the strategy mix stays stable (2/2)", y=1.04)
lfa.save_fig(fig2, "JSY084", "B1b_strategy_by_day", OUTPUT_ROOT)
plt.show()

In [ ]:
fig = lfa.fig_B2_strategy_late(E, "JSY084")
fig.suptitle("From saline onward, anticipation shifts to the screen transition", y=1.04)
lfa.save_fig(fig, "JSY084", "B2_strategy_late", OUTPUT_ROOT)
plt.show()

In [ ]:
fig = lfa.fig_B3_any_vr_lick(E, "JSY084")
fig.suptitle("Approach licking becomes rarer and concentrates at the reward zone", y=1.04)
lfa.save_fig(fig, "JSY084", "B3_any_vr_lick_inzone", OUTPUT_ROOT)
plt.show()

In [ ]:
fig_p = lfa.fig_persistence(E, "JSY084")
fig_p.suptitle("Drinking and dark-period anticipation are unchanged (1/2 — persistence)", y=1.04)
lfa.save_fig(fig_p, "JSY084", "B4a_post_vr_persistence", OUTPUT_ROOT)
plt.show()

fig_d = lfa.fig_dark_anticipation(E, "JSY084")
fig_d.suptitle("Drinking and dark-period anticipation are unchanged (2/2 — dark epoch)", y=1.04)
lfa.save_fig(fig_d, "JSY084", "B4b_dark_anticipation", OUTPUT_ROOT)
plt.show()

## 5. Comparison figure (C1)

z-scores on the M1 (B5-B8) basis; horizontal lines at z=0 and |z|=2. This is
a magnitude/direction comparison, not a significance test — see §2 above
for which of these actually have a calibration-passing bootstrap p-value
behind them (only `frac_approach_licks_in_rz` in JSY083 and `lick_spread_cm`
in JSY084 passed calibration at all; DCZ200's `frac_approach_licks_in_rz`
drop in JSY083 is the one point on this whole figure with a trustworthy
p-value, p=0.028).

**Caution reading magnitudes here**: a handful of these z-scores are very
large (|z|>10) because the B5-B8 SD itself is small for that metric (4
days only) — a large |z| can reflect "genuinely large effect" or "unstable
small-sample SD," and this figure alone can't tell you which. Cross-check
against the raw values in the CSV export (§6) before quoting a specific
z-score.

In [ ]:
fig = lfa.fig_C1_comparison(E)
fig.suptitle("Shared and divergent DCZ effects", y=1.005)
lfa.save_fig(fig, "comparison", "C1_shared_divergent_effects", OUTPUT_ROOT)
plt.show()

## 6. Per-animal CSV export

In [ ]:
for a in lfa.ANIMALS:
    out = lfa.export_animal_csv(E, a, OUTPUT_ROOT)
    print("wrote", out)
E

---
### What changed vs. the original `3c_*` figures — quick reference

- **M2 calibration**: only 2 of 6 (animal x metric) bootstrap tests pass
  calibration at all (JSY083 `frac_approach_licks_in_rz`, JSY084
  `lick_spread_cm`); `run_speed_vr_cms` fails badly in BOTH animals
  (FPR=0.50) — this metric's day-to-day variability is too large for
  reliable trial-level inference with only 4 reference days, confirming
  the critique's concern. Only one p-value in the whole notebook survives:
  JSY083 DCZ200 `frac_approach_licks_in_rz`, p=0.028.
- **M1+M7 correction**: with the B5-B8 baseline and the corridor-start
  carryover filter, JSY083's `lick_spread_cm` at DCZ200 flips from the
  earlier POSITIVE z (+2.6, on the B7+B8/unfiltered basis) to NEGATIVE
  (z≈-1.9) — that earlier finding does not survive the fix.
- **M4 correction**: the actual post-valve-to-next-teleport gap is ~1.0s,
  not the nominal 1.5s the original persistence metric assumed — the
  corrected (exact-timestamp) persistence fractions are substantially
  higher than the nominal-1.5s version for every session.
- **M3**: restricting "spatial" to the final 30cm reduces the spatial-
  strategy fraction across the board (e.g. JSY083 B1: 0.41 -> 0.26) but
  the qualitative baseline-learning pattern (JSY083 declining, JSY084 flat)
  is unchanged.